In [ ]:
# =========================================================
# Automated EDA Assistant — Setup
# =========================================================
# (Chạy cell này trước để cài đặt các thư viện)

# ---- Optional installs (bật/tắt các flag dưới) ----
USE_PROFILING = True      # Tạo báo cáo ydata-profiling (HTML)
USE_LLM_INSIGHT = False   # Sinh insight tự động bằng LLM (OpenRouter/OpenAI)

# Cài thư viện khi cần
try:
    import pandas as pd, numpy as np
    import matplotlib.pyplot as plt, seaborn as sns
    import scipy
except Exception:
    !pip -q install pandas numpy matplotlib seaborn scipy

if USE_PROFILING:
    try:
        import ydata_profiling
    except Exception:
        !pip -q install ydata-profiling

# LLM insight (tuỳ chọn)
if USE_LLM_INSIGHT:
    # Bạn có thể dùng 1 trong 2: openai (OpenAI) hoặc httpx (OpenRouter)
    try:
        import openai
    except Exception:
        !pip -q install openai
    try:
        import httpx
    except Exception:
        !pip -q install httpx

# Imports chung
import os, io, base64, textwrap, warnings
from datetime import datetime
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

SAVE_DIR = "/content/eda_report"
os.makedirs(SAVE_DIR, exist_ok=True)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
sns.set_style("whitegrid")

print("Environment ready. SAVE_DIR =", SAVE_DIR)


Environment ready. SAVE_DIR = /content/eda_report


In [ ]:
# =========================================================
# Data Loading
# =========================================================
# Upload trực tiếp file CSV/Excel từ máy
# Tải data ở link này: https://drive.google.com/file/d/1WuMh4t0_KwH5en7SPaLiM1UtU0vxJK70/view?usp=sharing

USE_UPLOAD = True   # True -> bật upload; False -> dùng DATA_PATH

DATA_PATH = ""      # Điền đường dẫn nếu USE_UPLOAD = False, ví dụ: "/content/sales.csv"

TARGET_COL   = None     # Điền tên cột target nếu có (vd: "Sales")
DATE_COL     = None     # Điền tên cột ngày/giờ nếu có (vd: "Date")
ID_COLS      = []       # Điền các cột định danh nên bỏ khỏi EDA nếu cần (vd: ["id"])

# Nếu không có sẵn dữ liệu, tạo 1 dataset mẫu để thử
CREATE_SAMPLE_IF_EMPTY = True

def load_dataframe():
    df = None
    if USE_UPLOAD:
        try:
            from google.colab import files
            uploaded = files.upload()
            fname = list(uploaded.keys())[0]
            if fname.lower().endswith((".xlsx", ".xls")):
                df = pd.read_excel(io.BytesIO(uploaded[fname]))
            else:
                df = pd.read_csv(io.BytesIO(uploaded[fname]))
            print(f"Loaded uploaded file: {fname} → shape={df.shape}")
        except Exception as e:
            print(" Upload failed or not in Colab. Error:", e)
    if df is None and DATA_PATH:
        if DATA_PATH.lower().endswith((".xlsx", ".xls")):
            df = pd.read_excel(DATA_PATH)
        else:
            df = pd.read_csv(DATA_PATH)
        print(f" Loaded from path: {DATA_PATH} → shape={df.shape}")
    if df is None and CREATE_SAMPLE_IF_EMPTY:
        rng = np.random.default_rng(42)
        n = 500
        df = pd.DataFrame({
            "Date": pd.date_range("2024-01-01", periods=n, freq="D"),
            "Region": rng.choice(["North", "South", "East", "West"], size=n),
            "Channel": rng.choice(["Online", "Retail", "Distributor"], size=n),
            "Customers": rng.integers(10, 300, size=n),
            "Orders": rng.integers(1, 80, size=n),
            "UnitPrice": rng.normal(20, 4, size=n).round(2),
        })
        df["Sales"] = (df["Orders"] * df["UnitPrice"] * rng.uniform(0.8, 1.2, size=n)).round(2)
        print("ℹ No file provided → Created a synthetic sample dataset:", df.shape)
    return df

df = load_dataframe()

# Chuẩn hoá kiểu dữ liệu cơ bản
def coerce_types(df, date_col=DATE_COL):
    dfc = df.copy()
    # Parse date column nếu có
    if date_col and date_col in dfc.columns:
        dfc[date_col] = pd.to_datetime(dfc[date_col], errors="coerce")
    # Thử ép kiểu numeric với các cột object có thể chuyển
    for col in dfc.columns:
        if dfc[col].dtype == "object":
            # Nếu là dạng số có dấu phẩy/ngăn cách -> làm sạch nhẹ
            dfc[col] = dfc[col].astype(str).str.replace(",", "").str.strip()
            dfc[col] = pd.to_numeric(dfc[col], errors="ignore")
    return dfc

df = coerce_types(df)
print(" Columns:", list(df.columns))
df.head(3)


Saving Top 50 Animation Movies and TV Shows.csv to Top 50 Animation Movies and TV Shows.csv
Loaded uploaded file: Top 50 Animation Movies and TV Shows.csv → shape=(50, 7)
 Columns: ['Ranking', 'Name', 'Year', 'Minutes', 'genre', 'Rating', 'Votes']


,Ranking,Name,Year,Minutes,genre,Rating,Votes
0,1,Big Mouth,(2017– ),30 min,Animation Comedy Romance,7.9,79301
1,2,The Bad Guys,-2022,100 min,Animation Adventure Comedy,6.8,37335
2,3,Chainsaw Man,(2022– ),nan,Animation Action Adventure,8.8,10613


In [ ]:
# =========================================================
# Utils: save figures, tables, simple HTML report
# =========================================================

def _safe_name(s: str) -> str:
    return "".join(c if c.isalnum() else "_" for c in str(s))[:64]

def save_fig(path):
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight", dpi=150)
    plt.close()

def save_table_csv(df_table, name):
    path = os.path.join(SAVE_DIR, f"{_safe_name(name)}.csv")
    df_table.to_csv(path, index=True)
    return path

def embed_img_tag(img_path, width="700px"):
    rel = os.path.relpath(img_path, SAVE_DIR)
    return f'<img src="{rel}" width="{width}" style="margin:10px 0;">'

def write_html_report(title, sections):
    """
    sections: list of dicts with keys:
      - heading: str
      - html: str (raw html)
    """
    now = datetime.now().strftime("%Y-%m-%d %H:%M")
    css = """
    body { font-family: Arial, sans-serif; margin: 24px; color:#222;}
    h1 { font-size: 24px; margin-bottom: 0; }
    .sub { color:#777; margin: 4px 0 16px 0; }
    h2 { font-size: 20px; margin-top: 28px; }
    table { border-collapse: collapse; margin: 10px 0; }
    th, td { border: 1px solid #ddd; padding: 6px 8px; }
    .card { background:#fafafa; border:1px solid #eee; padding:14px; border-radius:8px; }
    a { color:#0a66c2; text-decoration:none; }
    """
    html = [f"<html><head><meta charset='utf-8'><style>{css}</style></head><body>"]
    html += [f"<h1>{title}</h1><div class='sub'>Generated at {now}</div><hr>"]
    for sec in sections:
        html += [f"<h2>{sec['heading']}</h2>"]
        html += [f"<div class='card'>{sec['html']}</div>"]
    html += ["</body></html>"]
    out = os.path.join(SAVE_DIR, "index.html")
    with open(out, "w", encoding="utf-8") as f:
        f.write("\n".join(html))
    return out

print("Utils ready.")


Utils ready.


In [ ]:
# =========================================================
# Overview, Missing values, Duplicates, Types
# =========================================================

def dataset_overview(df):
    info = {
        "rows": df.shape[0],
        "cols": df.shape[1],
        "memory_mb": round(df.memory_usage(deep=True).sum() / (1024**2), 2)
    }
    dtypes = df.dtypes.astype(str).value_counts().rename_axis("dtype").reset_index(name="count")
    missing = df.isna().sum().to_frame("missing").assign(pct=lambda x: (x["missing"]/len(df)*100).round(2))
    dup_rows = int(df.duplicated().sum())

    # Save tables
    path_missing = save_table_csv(missing, "missing_summary")
    path_dtypes  = save_table_csv(dtypes.set_index("dtype"), "dtype_counts")
    # Small HTML snippet
    html = f"""
      <b>Shape:</b> {info['rows']} rows × {info['cols']} cols<br>
      <b>Memory:</b> {info['memory_mb']} MB<br>
      <b>Duplicate rows:</b> {dup_rows}<br>
      <b>Type counts CSV:</b> {_safe_name('dtype_counts')}.csv<br>
      <b>Missing summary CSV:</b> {_safe_name('missing_summary')}.csv
    """
    return {"info": info, "missing": missing, "dtypes": dtypes, "dup_rows": dup_rows, "html": html}

overview = dataset_overview(df)
print("Overview:", overview["info"])
display(df.head(5))


Overview: {'rows': 50, 'cols': 7, 'memory_mb': np.float64(0.02)}


,Ranking,Name,Year,Minutes,genre,Rating,Votes
0,1,Big Mouth,(2017– ),30 min,Animation Comedy Romance,7.9,79301
1,2,The Bad Guys,-2022,100 min,Animation Adventure Comedy,6.8,37335
2,3,Chainsaw Man,(2022– ),nan,Animation Action Adventure,8.8,10613
3,4,Rick and Morty,(2013– ),23 min,Animation Adventure Comedy,9.1,515315
4,5,Disenchanted,-2022,nan,Animation Adventure Comedy,NaN,nan


In [ ]:
# =========================================================
# Numeric & Categorical summaries + plots
# =========================================================
NUM_TOP_PLOTS = 12  # tối đa bao nhiêu cột numeric để vẽ

def numeric_summary(df):
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    desc = df[num_cols].describe().T if num_cols else pd.DataFrame()
    if not desc.empty:
        desc["missing"] = df[num_cols].isna().sum()
        desc["skew"] = df[num_cols].skew(numeric_only=True)
        desc["kurt"] = df[num_cols].kurtosis(numeric_only=True)
    path_desc = save_table_csv(desc, "numeric_describe")
    return num_cols, desc, path_desc

def plot_numeric_distributions(df, num_cols, top=NUM_TOP_PLOTS):
    if not num_cols:
        return []
    # chọn theo variance lớn nhất
    var_rank = df[num_cols].var().sort_values(ascending=False).index.tolist()[:top]
    img_paths = []
    for col in var_rank:
        fig, ax = plt.subplots(figsize=(8,4))
        sns.histplot(data=df, x=col, bins=30, kde=True, ax=ax)
        ax.set_title(f"Distribution — {col}")
        p = os.path.join(SAVE_DIR, f"hist_{_safe_name(col)}.png")
        save_fig(p); img_paths.append(p)
    return img_paths

def categorical_summary(df, top_k_cols=8, top_k_values=15):
    cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    freq_tables = {}
    img_paths = []
    # lấy các cột có số lượng unique hợp lý
    counts = {c: df[c].nunique(dropna=True) for c in cat_cols}
    cat_rank = sorted(cat_cols, key=lambda c: counts[c], reverse=True)[:top_k_cols]
    for col in cat_rank:
        vc = df[col].value_counts(dropna=False).head(top_k_values)
        freq_tables[col] = vc
        # plot
        fig, ax = plt.subplots(figsize=(9,4))
        vc.plot(kind="bar", ax=ax)
        ax.set_title(f"Top {top_k_values} values — {col}")
        ax.set_ylabel("count")
        p = os.path.join(SAVE_DIR, f"bar_{_safe_name(col)}.png")
        save_fig(p); img_paths.append(p)
    # lưu các bảng tần suất
    for col, vc in freq_tables.items():
        save_table_csv(vc.to_frame("count"), f"freq_{col}")
    return cat_cols, freq_tables, img_paths

num_cols, num_desc, path_num_desc = numeric_summary(df)
num_imgs = plot_numeric_distributions(df, num_cols)
cat_cols, cat_freqs, cat_imgs = categorical_summary(df)

print(f" Numeric columns: {len(num_cols)}; Categorical columns: {len(cat_cols)}")


 Numeric columns: 2; Categorical columns: 5


In [ ]:
# =========================================================
# Outlier detection (IQR) — per numeric column
# =========================================================
def iqr_outlier_bounds(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    low  = q1 - k*iqr
    high = q3 + k*iqr
    return low, high

def outlier_report(df, num_cols, k=1.5):
    if not num_cols:
        return pd.DataFrame(), pd.DataFrame()
    bounds = []
    masks = []
    for col in num_cols:
        series = df[col].dropna()
        if series.empty:
            continue
        low, high = iqr_outlier_bounds(series, k=k)
        out_mask = (df[col] < low) | (df[col] > high)
        masks.append(out_mask.rename(col))
        bounds.append({"column": col, "low": low, "high": high, "outliers": int(out_mask.sum())})
    bounds_df = pd.DataFrame(bounds).set_index("column")
    # hàng có ít nhất 1 outlier
    if masks:
        any_outlier = pd.concat(masks, axis=1).any(axis=1)
        out_rows = df[any_outlier].copy()
    else:
        out_rows = pd.DataFrame()
    return bounds_df, out_rows

bounds_df, out_rows = outlier_report(df, num_cols, k=1.5)
path_bounds = save_table_csv(bounds_df, "outlier_bounds_iqr")
path_outrows = save_table_csv(out_rows, "outlier_rows_iqr")

print(f" Outlier summary saved: {path_bounds}")
print(f" Outlier rows saved: {path_outrows} (rows={len(out_rows)})")


 Outlier summary saved: /content/eda_report/outlier_bounds_iqr.csv
 Outlier rows saved: /content/eda_report/outlier_rows_iqr.csv (rows=1)


In [ ]:
# =========================================================
# Correlation analysis
# =========================================================
def correlation_analysis(df, num_cols, method="pearson", top_k_pairs=20):
    if not num_cols or len(num_cols) < 2:
        return None, pd.DataFrame(), []
    corr = df[num_cols].corr(method=method)
    # heatmap
    fig, ax = plt.subplots(figsize=(10,8))
    sns.heatmap(corr, ax=ax, cmap="coolwarm", center=0, square=True)
    ax.set_title(f"{method.title()} correlation heatmap")
    heatmap_path = os.path.join(SAVE_DIR, f"corr_heatmap_{method}.png")
    save_fig(heatmap_path)

    # top abs correlation pairs
    c = corr.abs().where(~np.eye(corr.shape[0], dtype=bool))
    pairs = (c.stack().sort_values(ascending=False)
               .reset_index().rename(columns={"level_0":"var1","level_1":"var2",0:"abs_corr"}))
    pairs = pairs.drop_duplicates(subset=["abs_corr"])[:top_k_pairs]
    path_pairs = save_table_csv(pairs.set_index(["var1","var2"]), f"top_corr_pairs_{method}")

    return heatmap_path, pairs, path_pairs

heatmap_png, corr_pairs, path_corr_pairs = correlation_analysis(df, num_cols, method="pearson", top_k_pairs=30)
print(" Correlation analysis done.")
corr_pairs.head(10)

#thêm insight
def generate_corr_insights(pairs, threshold=0.5):
    insights = []

    for _, row in pairs.iterrows():
        v1, v2, corr = row["var1"], row["var2"], row["abs_corr"]

        if corr >= threshold:
            insights.append(f"{v1} & {v2}: strong correlation ({corr:.2f})")
        elif corr >= 0.3:
            insights.append(f"{v1} & {v2}: moderate correlation ({corr:.2f})")

    return insights

 # tạo insight
corr_insights = generate_corr_insights(corr_pairs)

# in ra
for i in corr_insights[:5]:
    print("-", i)


 Correlation analysis done.


In [ ]:
# phân tích giữa biến số và biến chữ (chỉ chọn 1 biến target)
#num ↔ cat → ANOVA / Kruskal
#ordinal ↔ cat → Kruskal
#auto insight text

# =========================================================
# NUM vs CAT ANALYSIS (FINAL FIX)
# =========================================================

import pandas as pd
import numpy as np
import scipy.stats as stats

# =========================================================
# 1. CLEAN COLUMN
# =========================================================
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("✅ Columns:", df.columns.tolist())


# =========================================================
# 2. AUTO CONVERT NUMERIC
# =========================================================
for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass


# =========================================================
# 3. AUTO DETECT MULTI TARGET (SURVEY)
# =========================================================
def detect_targets(df):

    # ưu tiên pattern a1, a2, b1...
    targets = [col for col in df.columns if col.startswith(("a", "b", "c"))]

    # fallback nếu không có
    if not targets:
        targets = [
            col for col in df.columns
            if any(x in col for x in ["score", "rating", "hài", "satisfaction"])
        ]

    return targets


target_cols = detect_targets(df)

print("🎯 Targets:", target_cols)


# =========================================================
# 4. AUTO DETECT CAT COLS
# =========================================================
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

# loại target nếu lỡ nằm trong cat
cat_cols = [c for c in cat_cols if c not in target_cols]

print("📊 Cat cols:", cat_cols)


# =========================================================
# 5. ANALYSIS FUNCTION
# =========================================================
def analyze_cat_vs_num(df, cat_col, num_col):

    data = df[[cat_col, num_col]].dropna()

    if data.empty:
        return None

    groups = data[cat_col].unique()
    samples = [data[data[cat_col] == g][num_col] for g in groups]

    samples = [s for s in samples if len(s) >= 3]
    if len(samples) < 2:
        return None

    # normality
    normal = True
    for s in samples:
        try:
            _, p = stats.shapiro(s)
            if p < 0.05:
                normal = False
                break
        except:
            normal = False

    # test
    try:
        if normal:
            _, p = stats.f_oneway(*samples)
            test = "ANOVA"
        else:
            _, p = stats.kruskal(*samples)
            test = "Kruskal"
    except:
        return None

    medians = data.groupby(cat_col)[num_col].median().sort_values(ascending=False)

    if p < 0.05:
        insight = f"{cat_col} ảnh hưởng đến {num_col} (p={p:.4f}), nhóm cao nhất: {medians.index[0]}"
    else:
        insight = f"{cat_col} không ảnh hưởng đáng kể (p={p:.4f})"

    return {
        "feature": cat_col,
        "test": test,
        "p_value": round(p, 5),
        "top_group": medians.index[0],
        "insight": insight
    }


# =========================================================
# 6. RUN ANALYSIS
# =========================================================
results = []

for target in target_cols:
    for col in cat_cols:
        try:
            res = analyze_cat_vs_num(df, col, target)
            if res:
                res["target"] = target   # 👈 thêm dòng này
                results.append(res)
        except:
            continue

mixed_df = pd.DataFrame(results)

if not mixed_df.empty:
    mixed_df = mixed_df.sort_values("p_value").reset_index(drop=True)

print("\n📊 RESULT:")
print(mixed_df)

✅ Columns: ['khối_làm_việc', 'khoa/_phòng', 'số_năm_công_tác_tại_bệnh_viện', 'số_năm_công_tác_tại_khoa', 'số_giờ_làm_việc', 'vị_trí_công_tác', 'vị_trí_khác', 'tiếp_xúc_với_bệnh_nhân', 'a1', 'a2', 'a3', 'a4', 'a5*', 'a6', 'a7*', 'a8*', 'a9', 'a10*', 'a11', 'a12*', 'a13', 'a14*', 'a15', 'a16*', 'a17*', 'a18', 'b1', 'b2', 'b3*', 'b4*', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6*', 'd1', 'd2', 'd3', 'd4', 'e1*', 'f1', 'f2*', 'f3*', 'f4', 'f5*', 'f6*', 'f7', 'f8', 'f9', 'f10*', 'f11', 'f12*', 'f13', 'f14*']
🎯 Targets: ['a1', 'a2', 'a3', 'a4', 'a5*', 'a6', 'a7*', 'a8*', 'a9', 'a10*', 'a11', 'a12*', 'a13', 'a14*', 'a15', 'a16*', 'a17*', 'a18', 'b1', 'b2', 'b3*', 'b4*', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6*']
📊 Cat cols: ['khối_làm_việc', 'khoa/_phòng', 'số_năm_công_tác_tại_bệnh_viện', 'số_năm_công_tác_tại_khoa', 'số_giờ_làm_việc', 'vị_trí_công_tác', 'vị_trí_khác', 'tiếp_xúc_với_bệnh_nhân']

📊 RESULT:
                    feature     test  p_value  \
0           số_giờ_làm_việc  Kruskal  0.00000   
1      

In [ ]:
# =========================================================
# AUTO EDA (SURVEY + VISUAL) mở rộng phân tích nhiều biến chữ vs biến số
# =========================================================

import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import os

SAVE_DIR = "./eda_output"
PLOT_DIR = os.path.join(SAVE_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

# ===== 1. CLEAN =====
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# ===== 2. CONVERT NUMERIC =====
for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass

# ===== 3. DETECT TARGETS (👉 sửa nếu cần)
target_cols = [c for c in df.columns if c.startswith(("a","b","c"))]

# fallback
if not target_cols:
    target_cols = df.select_dtypes(include=np.number).columns.tolist()

print("🎯 Targets:", target_cols)

# ===== 4. CAT COLS =====
cat_cols = df.select_dtypes(include=["object","category"]).columns.tolist()
cat_cols = [c for c in cat_cols if c not in target_cols]

print("📊 Cat cols:", cat_cols)

# =========================================================
# FUNCTION TEST
# =========================================================
def analyze(df, cat, num):

    data = df[[cat,num]].dropna()
    if data.empty: return None

    groups = data[cat].unique()
    samples = [data[data[cat]==g][num] for g in groups]
    samples = [s for s in samples if len(s)>=3]
    if len(samples)<2: return None

    normal=True
    for s in samples:
        try:
            if stats.shapiro(s)[1]<0.05:
                normal=False; break
        except: normal=False

    try:
        if normal:
            p = stats.f_oneway(*samples)[1]; test="ANOVA"
        else:
            p = stats.kruskal(*samples)[1]; test="Kruskal"
    except:
        return None

    med = data.groupby(cat)[num].median().sort_values(ascending=False)

    insight = (f"{cat} ảnh hưởng {num} (p={p:.4f}), nhóm cao nhất: {med.index[0]}"
               if p<0.05 else
               f"{cat} không ảnh hưởng {num} (p={p:.4f})")

    return {"feature":cat,"target":num,"test":test,"p_value":round(p,5),
            "top_group":med.index[0],"insight":insight}


# =========================================================
# FUNCTION PLOT
# =========================================================
def plot(df, cat, num):

    data = df[[cat,num]].dropna()
    if data.empty: return None,None

    # box
    plt.figure(figsize=(5,3))
    sns.boxplot(x=cat,y=num,data=data)
    plt.xticks(rotation=30)
    box = os.path.join(PLOT_DIR,f"box_{cat}_{num}.png")
    plt.tight_layout(); plt.savefig(box); plt.close()

    # bar median
    med = data.groupby(cat)[num].median().reset_index()
    plt.figure(figsize=(5,3))
    sns.barplot(x=cat,y=num,data=med)
    plt.xticks(rotation=30)
    bar = os.path.join(PLOT_DIR,f"bar_{cat}_{num}.png")
    plt.tight_layout(); plt.savefig(bar); plt.close()

    return box, bar


# =========================================================
# RUN
# =========================================================
rows=[]

for t in target_cols:
    for c in cat_cols:
        try:
            r = analyze(df,c,t)
            if r:
                if r["p_value"]<0.05:  # chỉ vẽ cái quan trọng
                    b1,b2 = plot(df,c,t)
                else:
                    b1,b2 = None,None

                r["boxplot"]=b1
                r["barplot"]=b2
                rows.append(r)
        except:
            continue

result = pd.DataFrame(rows)

if not result.empty:
    result = result.sort_values("p_value").reset_index(drop=True)

print("\n📊 RESULT:")
print(result.head(10))


# =========================================================
# SUMMARY (🔥 QUAN TRỌNG)
# =========================================================
important = result[result["p_value"]<0.05]

summary = (important.groupby("feature")
           .size()
           .sort_values(ascending=False))

print("\n🏆 TOP DRIVERS:")
print(summary.head(10))

🎯 Targets: ['ranking', 'rating']
📊 Cat cols: ['name', 'year', 'minutes', 'genre', 'votes']

📊 RESULT:
   feature   target     test  p_value                  top_group  \
0  minutes   rating    ANOVA  0.00273                      7 min   
1     year   rating  Kruskal  0.00805                (2005–2008)   
2    genre   rating  Kruskal  0.01131           Animation Family   
3    genre  ranking    ANOVA  0.08234  Animation Adventure Drama   
4  minutes  ranking  Kruskal  0.35592                     50 min   
5     year  ranking    ANOVA  0.91908                      -1994   

                                             insight  \
0  minutes ảnh hưởng rating (p=0.0027), nhóm cao ...   
1  year ảnh hưởng rating (p=0.0080), nhóm cao nhấ...   
2  genre ảnh hưởng rating (p=0.0113), nhóm cao nh...   
3           genre không ảnh hưởng ranking (p=0.0823)   
4         minutes không ảnh hưởng ranking (p=0.3559)   
5            year không ảnh hưởng ranking (p=0.9191)   

                            

In [ ]:
# =========================================================
# Auto profiling reports
# =========================================================

if USE_PROFILING:
    try:
        from ydata_profiling import ProfileReport
        prof = ProfileReport(df, title="EDA Data Report", minimal=True, explorative=True)
        prof_path = os.path.join(SAVE_DIR, "EDA_data_report.html")
        prof.to_file(prof_path)
        print("EDA_data_report:", prof_path)
    except Exception as e:
        print("EDA_data_report failed:", e)

        # =========================================================
# CUSTOM EDA INSIGHTS REPORT
# =========================================================

from jinja2 import Template

# ===== 1. AUTO CONCLUSION =====
def generate_conclusion(corr_pairs, mixed_df):

    lines = []

    if not mixed_df.empty:
        top = mixed_df.iloc[0]
        lines.append(
            f"'{top['feature']}' là yếu tố ảnh hưởng mạnh nhất "
            f"(p={top['p_value']:.4f}), nhóm '{top['top_group']}' cao nhất."
        )

    strong_corr = corr_pairs[corr_pairs["abs_corr"] >= 0.5]
    if not strong_corr.empty:
        c = strong_corr.iloc[0]
        lines.append(
            f"Tương quan mạnh giữa '{c['var1']}' và '{c['var2']}' "
            f"(corr={c['abs_corr']:.2f})."
        )

    if not lines:
        lines.append("Không phát hiện mối quan hệ đáng kể.")

    return " ".join(lines)


# ===== 2. GENERATE HTML REPORT =====
def generate_custom_report(corr_pairs, corr_insights, mixed_df, heatmap_path):

    conclusion = generate_conclusion(corr_pairs, mixed_df)

    # Top 3 feature
    if not mixed_df.empty:
        top_html = mixed_df.head(3).to_html(index=False)
    else:
        top_html = "<p>Không có yếu tố nổi bật</p>"

    html_template = """
    <html>
    <head>
        <title>EDA Insights Report</title>
        <style>
            body {font-family: Arial; margin: 40px;}
            h1 {color: #2c3e50;}
            h2 {color: #34495e;}
            table {border-collapse: collapse; width: 100%; margin-bottom: 20px;}
            th, td {border: 1px solid #ddd; padding: 8px;}
            th {background-color: #f2f2f2;}
            .box {background: #f9f9f9; padding: 15px; border-left: 5px solid #3498db;}
        </style>
    </head>

    <body>

    <h1>📊 EDA INSIGHTS REPORT</h1>

    <div class="box">
        <h2>🧠 Auto Conclusion</h2>
        <p>{{ conclusion }}</p>
    </div>

    <h2>🔥 Top 3 yếu tố ảnh hưởng</h2>
    {{ top_table }}

    <h2>📊 Correlation Heatmap</h2>
    <img src="{{ heatmap_path }}" width="600">

    <h2>📈 Correlation Insights</h2>
    <ul>
    {% for i in corr_insights %}
        <li>{{ i }}</li>
    {% endfor %}
    </ul>

    <h2>📊 Cat vs Num Analysis</h2>
    {{ mixed_table }}

    <h2>📎 Full Profiling Report</h2>
    <p><a href="EDA_data_report.html">Click để xem report chi tiết</a></p>

    </body>
    </html>
    """

    template = Template(html_template)

    html = template.render(
        conclusion=conclusion,
        top_table=top_html,
        heatmap_path=heatmap_png,
        corr_insights=corr_insights,
        mixed_table=mixed_df.to_html(index=False)
    )

    save_path = os.path.join(SAVE_DIR, "eda_custom_report.html")

    with open(save_path, "w", encoding="utf-8") as f:
        f.write(html)

    print("✅ Custom EDA report:", save_path)


# =========================================================
# RUN CUSTOM REPORT (PHẢI CÓ CÁC BIẾN TRƯỚC ĐÓ)
# =========================================================

try:
    generate_custom_report(
        corr_pairs=corr_pairs,
        corr_insights=corr_insights,
        mixed_df=mixed_df,
        heatmap_path=heatmap_png
    )
except Exception as e:
    print("❌ Custom report failed:", e)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 55/55 [00:00<00:00, 87.83it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

EDA_data_report: ./eda_output/EDA_data_report.html
✅ Custom EDA report: ./eda_output/eda_custom_report.html


In [ ]:
# Auto profiling reports nâng cao biểu đồ
# =========================================================

if USE_PROFILING:
    try:
        from ydata_profiling import ProfileReport
        prof = ProfileReport(df, title="EDA Data Report", minimal=True, explorative=True)
        prof_path = os.path.join(SAVE_DIR, "EDA_data_report.html")
        prof.to_file(prof_path)
        print("EDA_data_report:", prof_path)
    except Exception as e:
        print("EDA_data_report failed:", e)

# =========================================================
# CUSTOM EDA INSIGHTS REPORT
# =========================================================
from jinja2 import Template
import os
import pandas as pd

def generate_html_report(result):

    # ===== FIX IMAGE PATH (QUAN TRỌNG) =====
    result["boxplot"] = result["boxplot"].apply(
        lambda x: os.path.relpath(x, SAVE_DIR) if pd.notnull(x) else x
    )

    result["barplot"] = result["barplot"].apply(
        lambda x: os.path.relpath(x, SAVE_DIR) if pd.notnull(x) else x
    )

    # ===== 1. TOP DRIVER =====
    important = result[result["p_value"] < 0.05]

    if not important.empty:
        top_driver = (
            important.groupby("feature")
            .size()
            .sort_values(ascending=False)
            .index[0]
        )
    else:
        top_driver = "Không có"

    # ===== 2. AUTO CONCLUSION =====
    if not important.empty:
        conclusion = f"Yếu tố '{top_driver}' ảnh hưởng mạnh nhất đến nhiều khía cạnh hài lòng."
    else:
        conclusion = "Không phát hiện yếu tố ảnh hưởng rõ ràng."

    # ===== 3. GROUP DATA =====
    grouped = {}
    for _, r in result.iterrows():
        grouped.setdefault(r["target"], []).append(r)

    # ===== 4. HTML =====
    html_template = """
    <html>
    <head>
        <title>EDA Report</title>
        <style>
            body {font-family: Arial; margin: 40px;}
            h1 {color: #2c3e50;}
            h2 {color: #34495e;}
            .box {background: #f9f9f9; padding: 15px; margin-bottom:20px;}
            .sig {color: red; font-weight: bold;}
        </style>
    </head>

    <body>

    <h1>📊 EDA SURVEY REPORT</h1>

    <h2>🧠 Conclusion</h2>
    <p>{{ conclusion }}</p>

    <h2>🔥 Top Driver</h2>
    <p>{{ top_driver }}</p>

    {% for target, group in data.items() %}
        <div class="box">
            <h2>🎯 {{ target }}</h2>

            {% for row in group %}
                <p>
                    <b>{{ row.feature }}</b> →
                    <span class="{{ 'sig' if row.p_value < 0.05 else '' }}">
                        {{ row.insight }}
                    </span>
                </p>

                {% if row.boxplot %}
                    <img src="{{ row.boxplot }}" width="300">
                {% endif %}

                {% if row.barplot %}
                    <img src="{{ row.barplot }}" width="300">
                {% endif %}

                <hr>
            {% endfor %}
        </div>
    {% endfor %}

    </body>
    </html>
    """

    template = Template(html_template)

    html = template.render(
        data=grouped,
        top_driver=top_driver,
        conclusion=conclusion
    )

    path = os.path.join(SAVE_DIR, "eda_full_report.html")
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)

    print("✅ HTML report:", path)


# ===== RUN HTML REPORT =====
try:
    generate_html_report(result)
except Exception as e:
    print("❌ HTML report failed:", e)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 7/7 [00:00<00:00, 190.36it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

EDA_data_report: ./eda_output/EDA_data_report.html
✅ HTML report: ./eda_output/eda_full_report.html
